# 🏛️ JanShakti.AI — Text Classifier & Sentiment Model Training

**Kaggle Notebook — GPU Accelerated (T4/P100/H100)**

This notebook trains two models:
1. **Complaint Text Classifier** — TF-IDF + SGDClassifier (7 civic categories)
2. **Civic Sentiment Analyzer** — Fine-tuned DistilBERT (positive/negative/neutral)

### Outputs:
- `classifier.pkl` → Place in `backend/ml/weights/`
- `sentiment_model/` → Place in `backend/ml/weights/sentiment_model/`

In [ ]:
# Install dependencies
!pip install -q scikit-learn==1.5.2 transformers==4.45.0 torch datasets accelerate joblib pandas numpy

---
## Part 1: Complaint Text Classifier
### TF-IDF + SGDClassifier — 7 Civic Categories

Categories: Water Supply, Roads & Potholes, Drainage, Electricity, Garbage & Sanitation, Safety & Security, Public Health

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib
import os

print('Libraries loaded ✅')

In [ ]:
# ============================================================
# TRAINING DATA — 20 samples per category × 7 categories = 140+
# Add more samples here for higher accuracy!
# ============================================================

training_data = {
    "Water Supply": [
        "Water pipe burst on main road near school",
        "No water supply for 3 days in our area",
        "Drinking water contaminated with brown color",
        "Water tanker not arriving since last week",
        "Borewell not working in sector 5",
        "Leaking pipe flooding the street near market",
        "Low water pressure in apartments since Monday",
        "Water supply timing changed without notice",
        "Sewage water mixing with drinking water supply",
        "Water meter broken, need replacement urgently",
        "Tap water smells bad since yesterday morning",
        "Underground pipeline leaking near children park",
        "Water tank overflow causing flooding in colony",
        "No water connection despite applying 6 months ago",
        "Water supply disconnected wrongly without notice",
        "Paani ka pipe toot gaya hai gali mein",
        "Teen din se paani nahi aa raha hamari colony mein",
        "Nala overflow ho raha hai ward 7 mein",
        "Pipeline burst near children playground causing flooding",
        "Water supply irregular only comes for 30 minutes daily",
        "Boring water has iron content making it red colored",
        "Community water tank not being cleaned since months",
        "Water pressure too low on upper floors of building",
        "Open water pipeline running through dirty drain area",
        "Hand pump broken in village center affecting 50 families",
    ],
    "Roads & Potholes": [
        "Big pothole on highway near toll plaza causing accidents",
        "Road completely damaged after monsoon rains this year",
        "Speed breaker too high causing vehicle damage",
        "Footpath broken and dangerous for senior citizens",
        "Road cave-in near bus stop endangering commuters",
        "Tar road melting in summer heat creating sticky mess",
        "No road dividers on busy intersection causing chaos",
        "Construction debris left on main road blocking traffic",
        "Road not repaired after digging for cable laying",
        "Potholes causing flat tires and accidents daily",
        "Damaged road near hospital blocking ambulance route",
        "Broken speed bumps need urgent repair",
        "Road surface cracked badly after heavy rain",
        "Uneven road causing accidents at night especially",
        "Sinkhole forming on main road near market area",
        "Sadak toot gayi hai ward 12 mein bahut kharab",
        "Gaddhe bahut hain school ke paas road pe",
        "Highway pe accident ho raha hai daily pothole se",
        "Road not asphalted since 2 years despite complaints",
        "Muddy road completely impassable during monsoon season",
        "Flyover approach road has dangerous cracks",
        "Colony internal roads never repaired by municipality",
        "Road marking paint completely faded at zebra crossing",
        "Cement road broken near temple causing accidents",
        "New road already cracking within 3 months of construction",
    ],
    "Drainage": [
        "Drainage blocked causing severe waterlogging in area",
        "Sewer line overflow in residential colony for days",
        "Manhole cover missing on main road very dangerous",
        "Stagnant water breeding mosquitoes spreading dengue",
        "Flood water not draining since 2 days after rain",
        "Open drain near school very dangerous for children",
        "Gutter overflowing in front of shop ruining business",
        "Drain cleaning not done this month by municipality",
        "Nallah blocked with garbage and plastic waste",
        "Waterlogging during rains destroying standing crops",
        "Drain overflow entering house basement daily",
        "Underground drain pipe cracked leaking sewage",
        "Monsoon drainage system completely non functional",
        "Open drain cover is major safety hazard for kids",
        "Sewage backing up into homes during every rain",
        "Nallah mein kachra bhara hua hai saaf karo",
        "Barish mein paani ghar mein aa jata hai always",
        "Nala saaf nahi hua abhi tak koi nahi sunta",
        "Drainage system completely collapsed in old city area",
        "Waterlogging on bus route causing daily commute problems",
        "Storm water drain completely choked with construction debris",
        "Drain flies and insects breeding in blocked sewer",
        "Gutter smell unbearable in entire neighborhood",
        "Rainwater harvesting drain connected wrongly to sewer",
        "Open manhole causing accidents at night no warning signs",
    ],
    "Electricity": [
        "Streetlight not working for 2 weeks despite complaint",
        "Power outage daily for 6 hours in summer heat",
        "Transformer damaged need immediate replacement",
        "Electric pole leaning dangerously near playground",
        "Loose wiring hanging from pole very dangerous",
        "Voltage fluctuation damaging electronic appliances",
        "Streetlight opposite school entrance is broken",
        "No electricity connection since registration 3 months ago",
        "Power cable fallen on ground after storm electrocution risk",
        "Short circuit in public transformer sparking at night",
        "Bijli nahi aa rahi 5 ghante se bahut garmi hai",
        "Street light kharab hai raste pe andhera hai",
        "Transformer mein aag lag gayi thi abhi bhi repair nahi",
        "Electric pole bent after storm blocking road",
        "No power backup arrangement in district hospital area",
        "Frequent power cuts in summer 8 hours daily",
        "Underground cable fault in entire colony no power",
        "Streetlight timer not working stays on all day",
        "Electricity meter showing inflated wrong reading",
        "Power outage in critical medical facility area ICU",
        "Three phase power not available for farmers irrigation",
        "Illegal hooking causing low voltage in neighborhood",
        "LED streetlights installed but half already not working",
        "Power line sagging dangerously low near terrace",
        "Smart meter installed but billing still incorrect",
    ],
    "Garbage & Sanitation": [
        "Garbage not collected for one full week stinking badly",
        "Open dump site near residential area health hazard",
        "Community dustbin overflowing with household waste",
        "No sweeping happening in our street since 10 days",
        "Garbage burning in empty lot causing air pollution",
        "Dead animal carcass on road not removed since 3 days",
        "Public toilet not cleaned properly very dirty",
        "Solid waste dumped illegally in empty plot near homes",
        "Garbage truck not coming to our area this entire month",
        "Overflowing community dustbin spreading diseases",
        "Kachra uthaya nahi gaya ek hafta se badbu aa rahi hai",
        "Safai karmchari nahi aate humare area mein kabhi",
        "Garbage dump right next to drinking water source",
        "Sweeping machine not being deployed in our ward",
        "Waste segregation not happening despite door to door",
        "Trash littered all around vegetable food market area",
        "Community garbage bin broken needs immediate replacement",
        "Medical hospital waste dumped in regular community bin",
        "E-waste dumped openly in public area near school",
        "No garbage pickup service on weekends despite request",
        "Construction and demolition waste blocking footpath",
        "Plastic waste choking storm water drains in area",
        "Sanitation worker absent for weeks no replacement sent",
        "Open defecation area near railway track needs toilet",
        "Wet waste composting pit overflowing foul smell",
    ],
    "Safety & Security": [
        "Stray dogs attacking children in residential colony",
        "No street lights making entire area unsafe at night",
        "Illegal construction blocking emergency exit route",
        "Crime rate increasing in our neighborhood daily",
        "No CCTV cameras installed at dangerous intersection",
        "Drunk driving accidents happening near school zone",
        "Abandoned building being used for illegal activities",
        "No police patrol in our area at night feel unsafe",
        "Harassment of women near college bus stand regularly",
        "Fire safety equipment missing in community hall building",
        "Traffic signals not working at major intersection",
        "Dangerous wild animals spotted near village school",
        "Broken boundary fence allowing trespassers into colony",
        "Unsafe pedestrian crossing near primary school gate",
        "Drug dealing activity reported in public park area",
        "Unregistered vehicles parked illegally on footpath",
        "Chain snatching incidents increasing near ATM area",
        "Eve teasing problem near women bus stand unreported",
        "No safety barriers near construction site on main road",
        "Open water body without safety fencing children drown risk",
        "Liquor shop opened near school violating norms",
        "Speed breakers missing near accident prone zone",
        "No fire station within 10 km radius of our area",
        "Loose overhead cables low hanging electrocution risk",
        "Illegal encroachment on footpath forcing pedestrians on road",
    ],
    "Public Health": [
        "Dengue outbreak reported in our ward 15 cases this week",
        "Primary health center has no medicines in stock",
        "Government hospital ward not cleaned properly dirty",
        "Vaccination drive cancelled without any prior notice",
        "Doctor not available at government clinic since Monday",
        "Ambulance not responding to emergency calls in area",
        "Food poisoning outbreak from community kitchen food",
        "Mosquito fogging not done this entire monsoon season",
        "Stagnant water near colony causing malaria outbreak",
        "No medical facility available within 5 km rural area",
        "Essential medicine shortage in PHC for past 2 weeks",
        "Anganwadi center has no nutrition supply for children",
        "Contaminated street food sold near school making kids sick",
        "Mental health counseling services not available anywhere",
        "TB patients not getting proper treatment at district hospital",
        "Blood bank always out of stock during emergencies",
        "Measles cases rising rapidly in urban slum area",
        "No ambulance service available for remote rural patients",
        "Hospital overcrowded with patients sleeping on floor",
        "Expired medicines being distributed at government clinic",
        "Water borne diseases increasing due to contaminated supply",
        "No female doctor available at PHC women suffer",
        "Rabies vaccine unavailable after dog bite incident",
        "Community health worker not visiting our area for checkup",
        "Malnutrition cases high among tribal children in block",
    ]
}

# Build dataset
texts, labels = [], []
for category, examples in training_data.items():
    texts.extend(examples)
    labels.extend([category] * len(examples))

print(f'Total samples: {len(texts)}')
print(f'Categories: {len(set(labels))}')
for cat in sorted(set(labels)):
    print(f'  {cat}: {labels.count(cat)} samples')

In [ ]:
# ============================================================
# TRAIN THE TEXT CLASSIFIER
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    stop_words='english',
    sublinear_tf=True,
)

X = vectorizer.fit_transform(texts)
y = np.array(labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SGD Classifier (fast, great for text)
model = SGDClassifier(
    loss='modified_huber',
    max_iter=2000,
    random_state=42,
    class_weight='balanced',
    alpha=1e-4,
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = (y_pred == y_test).mean()

print(f'\n{"="*50}')
print(f'  Model Accuracy: {accuracy:.2%}')
print(f'{"="*50}\n')
print(classification_report(y_test, y_pred))

# Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'\nCross-validation: {cv_scores.mean():.2%} (±{cv_scores.std():.2%})')

In [ ]:
# ============================================================
# SAVE CLASSIFIER MODEL
# ============================================================

os.makedirs('/kaggle/working/weights', exist_ok=True)

bundle = {
    'model': model,
    'vectorizer': vectorizer,
    'categories': sorted(set(labels)),
    'accuracy': accuracy,
}
joblib.dump(bundle, '/kaggle/working/weights/classifier.pkl')
print('✅ Classifier saved to /kaggle/working/weights/classifier.pkl')

# Quick test
test_texts = [
    'Water pipe burst on main road flooding everything',
    'Big pothole near school causing accidents daily',
    'Garbage not collected since one week stinking',
    'Streetlight broken no light at night unsafe area',
    'Dengue cases increasing hospital has no medicine',
]
for t in test_texts:
    features = vectorizer.transform([t.lower()])
    pred = model.predict(features)[0]
    proba = max(model.predict_proba(features)[0])
    print(f'  "{t[:50]}..." → {pred} ({proba:.0%})')

---
## Part 2: Civic Sentiment Analyzer
### Fine-tuned DistilBERT — 3 Classes (Positive / Negative / Neutral)

In [ ]:
import torch
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from torch.utils.data import Dataset
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# SENTIMENT TRAINING DATA
# 0=Positive, 1=Negative, 2=Neutral
# ============================================================

positive = [
    "Thank you for fixing the road so quickly great work",
    "The garbage collection has improved a lot recently",
    "Hospital services are much better now than before",
    "Water supply is regular and clean in our area now",
    "New streetlights installed making our area much safer",
    "Drainage problem resolved within 2 days excellent service",
    "Very happy with the fast response from ward office staff",
    "Road repair done properly this time thank you MLA sir",
    "Clean drinking water now available great improvement done",
    "Community park renovated beautifully children love it",
    "School building repainted and looks brand new and clean",
    "Public toilet cleaned regularly now very good maintenance",
    "New bridge construction is excellent quality workmanship",
    "Traffic management improved significantly less congestion",
    "Quick ambulance response saved my fathers life grateful",
    "Pothole filled within one day of complaint impressive",
    "Electricity transformer replaced quickly power restored",
    "Sanitation workers doing excellent daily cleaning job",
    "New water pipeline installation benefiting entire ward",
    "Government scheme benefits reaching deserving families now",
    "Ward officer personally visited and resolved our issue",
    "Smart city initiative making real difference in our area",
    "Online complaint system working very efficiently happy",
    "Mosquito fogging done regularly this monsoon no dengue",
    "Community meeting addressed all pending citizen concerns",
]

negative = [
    "Roads are terrible potholes everywhere no one cares",
    "No water supply for the third day in a row suffering",
    "Garbage dump creating unbearable stench near our homes",
    "Hospital has no medicines or doctors patients dying",
    "Complete darkness at night all streetlights broken",
    "Sewage overflowing into our homes for weeks disgusting",
    "Drainage completely blocked flooding every single monsoon",
    "Fake road repair contractor did nothing cheated government",
    "Water contamination making children seriously sick daily",
    "Government scheme benefits not reaching poor families corruption",
    "Corruption in public distribution system stealing rations",
    "No response to repeated complaints about broken dangerous road",
    "School infrastructure crumbling dangerously children at risk",
    "Power cuts 8 hours daily absolutely no improvement made",
    "Crime increasing rapidly police not patrolling our area",
    "Municipal corporation workers demanding bribe for services",
    "Filthy open drains causing disease outbreak in slum area",
    "Ambulance never arrives on time people dying on way",
    "Illegal construction destroying green belt no action taken",
    "Stray dog menace children bitten no animal control",
    "Election promises completely false nothing delivered angry",
    "Public toilets in pathetic condition women cannot use",
    "Expired medicines distributed at primary health center",
    "Encroachment on public land no eviction despite orders",
    "Contractor used substandard material road broke in month",
]

neutral = [
    "When will the road repair work begin in our ward",
    "Information needed about water supply schedule timing",
    "Requesting update on garbage collection timing today",
    "Need details about new electricity connection process",
    "What are the timings for vaccination drive this month",
    "How to apply for new residential water connection",
    "Query about property tax payment deadline date",
    "Asking about school admission process for next session",
    "Request for duplicate birth certificate copy",
    "Information needed about PM housing scheme eligibility",
    "Status update needed on complaint filed last week",
    "Where is the nearest government hospital location",
    "What documents needed for ration card application",
    "When is the next gram sabha meeting scheduled",
    "How to register complaint online on portal",
]

sent_texts = positive + negative + neutral
sent_labels = [0]*len(positive) + [1]*len(negative) + [2]*len(neutral)

print(f'Sentiment samples: {len(sent_texts)}')
print(f'  Positive: {len(positive)}, Negative: {len(negative)}, Neutral: {len(neutral)}')

In [ ]:
# ============================================================
# DATASET CLASS
# ============================================================

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, padding=True,
            max_length=max_length, return_tensors='pt'
        )
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

# Load tokenizer and model
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
sentiment_model = DistilBertForSequenceClassification.from_pretrained(
    model_name, num_labels=3
).to(device)

dataset = SentimentDataset(sent_texts, sent_labels, tokenizer)
print(f'Dataset created: {len(dataset)} samples')

In [ ]:
# ============================================================
# FINE-TUNE ON GPU
# ============================================================

output_dir = '/kaggle/working/weights/sentiment_model'
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy='epoch',
    fp16=torch.cuda.is_available(),  # Mixed precision on GPU
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=sentiment_model,
    args=training_args,
    train_dataset=dataset,
)

print('Training sentiment model...')
trainer.train()
print('\n✅ Training complete!')

In [ ]:
# ============================================================
# SAVE SENTIMENT MODEL
# ============================================================

sentiment_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

config = {
    'model_name': model_name,
    'num_labels': 3,
    'label_map': {0: 'positive', 1: 'negative', 2: 'neutral'},
    'samples': len(sent_texts),
}
with open(os.path.join(output_dir, 'training_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print(f'✅ Sentiment model saved to: {output_dir}')

# Quick test
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline('sentiment-analysis', model=output_dir, device=0 if torch.cuda.is_available() else -1)
label_map = {0: 'positive', 1: 'negative', 2: 'neutral'}

test_sentences = [
    'Thank you for fixing the road quickly',
    'Roads are pathetic nobody cares about us',
    'When will the garbage truck come today',
]
for s in test_sentences:
    r = classifier(s)[0]
    print(f'  "{s}" → {r["label"]} ({r["score"]:.0%})')

---
## 📦 Download Your Models

After running all cells, download from Kaggle Output:
- `/kaggle/working/weights/classifier.pkl` → Place in `backend/ml/weights/classifier.pkl`
- `/kaggle/working/weights/sentiment_model/` → Place entire folder in `backend/ml/weights/sentiment_model/`

Done! 🎉